In [ ]:
import itertools
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

SEED = 20260916

import os
DATA_DIR = next(p for p in [".", "..", "../.."] if os.path.exists(f"{p}/train.csv"))
train = pd.read_csv(f"{DATA_DIR}/train.csv")
validation = pd.read_csv(f"{DATA_DIR}/validation.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

feature_columns = [
    c for c in train.columns
    if c not in ["row_id", "target"]
]

y_train = train["target"].to_numpy()
y_validation = validation["target"].to_numpy()

X_train_raw = train[feature_columns].to_numpy()
X_validation_raw = validation[feature_columns].to_numpy()
X_test_raw = test[feature_columns].to_numpy()


def add_pair_features(X):
    A = X[:, :256]
    B = X[:, 256:]

    abs_diff = np.abs(A - B)

    return np.hstack([
        X,
        abs_diff,
    ])


X_train_pair = add_pair_features(X_train_raw)
X_validation_pair = add_pair_features(X_validation_raw)
X_test_pair = add_pair_features(X_test_raw)

feature_sets = {
    "raw": (
        X_train_raw,
        X_validation_raw,
        X_test_raw,
    ),
    "pair": (
        X_train_pair,
        X_validation_pair,
        X_test_pair,
    ),
}

print("Raw:", X_train_raw.shape)
print("Pair:", X_train_pair.shape)